# Transcribe a recording for text-to-speech training (GPU)

Runs `transcribe_tts.py` on Colab's free GPU: about 5-10 minutes for an hour of audio, instead of an hour on a laptop.

**Before you start**, upload the folder `urdu-tts-stt` to Google Drive so it sits at `MyDrive/urdu-tts/stt`. It must contain
`transcribe.py`, `transcribe_tts.py`, `prompt.txt`, `corrections.tsv`, `corrections.learned.tsv`, the audio file and
(optionally) the list of parts to leave out.

Then: *Runtime → Change runtime type → T4 GPU*, and *Runtime → Run all*. Allow access to Google Drive when asked.
Results are written to `MyDrive/urdu-tts/stt/transcripts`. If Colab disconnects, run all again: it continues where it stopped.

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, ('No GPU in this session. Use Runtime > Change runtime type > T4 GPU > Save, then Run all. '
                             'If no GPU is offered, the free GPU quota is used up; try again later.')
print('GPU:', gpu.stdout.strip())

## 1. Settings

In [ ]:
AUDIO = 'rida-sohaib.mp3'      # the recording, inside MyDrive/urdu-tts/stt
EXCLUDE = 'removed.txt'        # parts with other voices ('' if none)
TEST = False                   # True: only 2 minutes from 00:10:00, to check everything works

## 2. Connect Google Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
STT = '/content/drive/MyDrive/urdu-tts/stt'
import os
missing = [f for f in ['transcribe.py', 'transcribe_tts.py', AUDIO] + ([EXCLUDE] if EXCLUDE else []) if not os.path.exists(f'{STT}/{f}')]
assert not missing, f'Not found in {STT}: {missing}'
# faster-whisper's GPU engine (ctranslate2 4.6) needs CUDA 12 libraries; Colab ships CUDA 13
# ones, so install the CUDA 12 libraries from pip and point the loader at them.
!pip install -q "faster-whisper==1.2.1" "ctranslate2==4.6.2" "nvidia-cublas-cu12" "nvidia-cudnn-cu12==9.*"
import glob
cuda12 = glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib') + glob.glob('/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib')
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda12 + [os.environ.get('LD_LIBRARY_PATH', '')])
print('CUDA 12 libraries:', cuda12)

## 3. Transcribe

In [ ]:
args = f'"{AUDIO}" --out-dir transcripts --device cuda'
if EXCLUDE:
    args += f' --exclude "{EXCLUDE}"'
if TEST:
    args += ' --start 00:10:00 --duration 00:02:00'
%cd $STT
!python transcribe_tts.py $args

## 4. Results

Download the `transcripts` folder from Drive (or just `<name>_tts.srt` and `<name>_tts_review.txt`).
Proofread the SRT, then on the PC cut the clips into the text-to-speech project:

    powershell -ExecutionPolicy Bypass -File .\Transcribe-ForTTS.ps1 "C:\path	o	heudio.mp3" --export "C:\path	o\<name>_tts.srt"

In [ ]:
!ls -l "$STT/transcripts"
name = os.path.splitext(AUDIO)[0] + ('_section' if TEST else '')
!head -n 12 "$STT/transcripts/{name}_tts.srt"
!head -n 3 "$STT/transcripts/{name}_tts_review.txt"